# 6 — What the model attends to

Cross-attention weights are the closest thing a transformer has to an explicit
word alignment. They are also a genuinely useful debugging tool: a diffuse,
structureless map usually means the model has collapsed to a language-model
prior and is ignoring the source.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nmt.utils.io import project_root, read_json
from nmt.viz.style import use_style

use_style()
print("project root:", ROOT)

In [ ]:
import torch
from nmt.inference.search import DecodeConfig
from nmt.inference.translator import Translator

PREFERRED = ROOT / "artifacts" / "checkpoints" / "bpe_scratch" / "best_bleu.pt"

if PREFERRED.exists():
    CHECKPOINT = PREFERRED
else:
    candidates = sorted((ROOT / "artifacts" / "checkpoints").glob("*/best_bleu.pt"))
    if not candidates:
        raise SystemExit(
            "No checkpoint found. Train a model first:\n"
            "  python -m nmt.training.train --config configs/bpe_scratch.yaml"
        )
    CHECKPOINT = candidates[0]
    print(f"{PREFERRED.parent.name} not trained yet; using {CHECKPOINT.parent.name} "
          "-- its translations will be poor.")

translator = Translator.from_checkpoint(
    CHECKPOINT, decode_config=DecodeConfig(strategy="beam", beam_size=4)
)
print("device    :", translator.device)
print("vocabulary:", f"{translator.tokenizer.vocab_size:,}")

## A first translation, both ways

In [ ]:
for sentence, direction in [
    ("The red house on the corner belongs to my grandmother.", "en-es"),
    ("No s\u00e9 si podr\u00e9 ir a la fiesta ma\u00f1ana.", "es-en"),
]:
    result = translator.translate(sentence, direction)
    print(f"[{direction}] {sentence}")
    print(f"       -> {result.translation}")
    print(f"          ({result.num_source_tokens} source tokens, "
          f"{result.num_output_tokens} generated, {result.seconds * 1000:.0f} ms)\n")

## The alignment

The orange box on each row marks the source position the model attended to most
while generating that token. English–Spanish adjective/noun reordering ("red
house" → "casa roja") should show up as a crossing in the diagonal.

In [ ]:
from nmt.viz.attention_plots import plot_alignment
from IPython.display import Image

SENTENCE, DIRECTION = "The red house is very big.", "en-es"

result = translator.translate(SENTENCE, DIRECTION, return_attention=True)
print(result.translation)

out = ROOT / "reports" / "figures" / "attention_alignment"
plot_alignment(
    result.attention, result.source_tokens, result.output_tokens, out,
    title=f"Cross-attention: {SENTENCE}",
    subtitle="averaged over heads and layers",
)
Image(str(out.with_suffix(".png")), width=800)

## Individual heads specialise

Averaging hides the structure. Different heads pick up different jobs — one
tracks the previous token, another the sentence boundary, another the aligned
content word — and the mean of eight such heads looks blander than any of
them.

In [ ]:
from nmt.viz.attention_plots import plot_head_grid

# Re-run storing attention, then take the last decoder layer's cross-attention.
source = torch.tensor([translator._encode_source(SENTENCE, DIRECTION)],
                      device=translator.device)
generated = torch.tensor([[translator.tokenizer.piece_to_id(p)
                           for p in result.output_tokens]], device=translator.device)

with torch.no_grad():
    translator.model(source, generated[:, :-1], store_attention=True)

cross = translator.model.attention_maps()["cross"]
print(f"{len(cross)} decoder layers, each {tuple(cross[0].shape)} "
      "= (batch, heads, target_len, source_len)")

out = ROOT / "reports" / "figures" / "attention_heads"
plot_head_grid(
    cross[-1][0], result.source_tokens, result.output_tokens[:-1], out,
    title="Cross-attention per head, final decoder layer",
)
Image(str(out.with_suffix(".png")), width=900)

## Attention sharpens with depth

In [ ]:
from nmt.viz.attention_plots import plot_layer_progression

out = ROOT / "reports" / "figures" / "attention_layers"
plot_layer_progression(
    [layer[0] for layer in cross],
    result.source_tokens, result.output_tokens[:-1], out,
)
Image(str(out.with_suffix(".png")), width=900)

The mean entropy printed under each panel quantifies it: a lower value means
the distribution is more concentrated, i.e. the decoder has committed to a
specific source position.

## Decoding settings, live

Beam search usually buys 1–2 BLEU. The length penalty matters more than it
looks: without it, beam search systematically truncates, because every
log-probability is negative and longer hypotheses accumulate a worse score just
for being longer.

In [ ]:
SENTENCE = "She has been studying Spanish for three years and can now hold a conversation."

for strategy, beam, penalty in [
    ("greedy", 1, 0.0),
    ("beam", 4, 0.0),
    ("beam", 4, 0.6),
    ("beam", 8, 0.6),
    ("beam", 4, 1.2),
]:
    translator.decode_config = DecodeConfig(
        strategy=strategy, beam_size=beam, length_penalty=penalty
    )
    result = translator.translate(SENTENCE, "en-es")
    label = f"{strategy}" + (f" b={beam} a={penalty}" if strategy == "beam" else "")
    print(f"{label:18s} ({result.num_output_tokens:2d} tok) {result.translation}")

## Where it breaks

In [ ]:
translator.decode_config = DecodeConfig(strategy="beam", beam_size=4, length_penalty=0.6)

hard = [
    ("Tom bought a Volkswagen in Saskatchewan.", "en-es"),   # rare proper nouns
    ("I have 1,234 books and 56 magazines.", "en-es"),        # numbers
    ("The man who the woman that I saw yesterday met is my brother.", "en-es"),  # nesting
    ("Se lo di\u00f3.", "es-en"),                                  # clitics, ambiguous
    ("\u00a1Qu\u00e9 fr\u00edo hace hoy!", "es-en"),                          # exclamative
]
for sentence, direction in hard:
    result = translator.translate(sentence, direction)
    print(f"[{direction}] {sentence}\n       -> {result.translation}\n")